# Classify Job Postings

Run `3-analyse-aie-job-postings.ipynb` first so that `Job Posting Data/1-scraped_jobs.jsonl` exists.

This notebook reads the scraped jobs, uses an LLM to classify whether each role is truly an AI engineering role, and writes the accepted jobs to `Job Posting Data/2-llm_filtered_jobs.jsonl`.


In [ ]:
import json
from pathlib import Path

import pandas as pd
from openai import OpenAI
from IPython.display import HTML, display

screening_jsonl_path = Path("Job Posting Data") / "1-scraped_jobs.jsonl"
llm_filtered_jsonl_path = Path("Job Posting Data") / "2-llm_filtered_jobs.jsonl"

if not screening_jsonl_path.exists():
    raise FileNotFoundError(
        f"Scraped jobs JSONL file not found: {screening_jsonl_path.resolve()}. Run 3-analyse-aie-job-postings.ipynb first."
    )

screening_source_jobs = pd.read_json(screening_jsonl_path, lines=True)

if screening_source_jobs.empty:
    raise ValueError("The scraped jobs JSONL file is empty. Run 3-analyse-aie-job-postings.ipynb first.")

if "description" not in screening_source_jobs.columns:
    raise KeyError("The scraped jobs JSONL file must contain a 'description' column.")

screening_source_jobs = screening_source_jobs[screening_source_jobs["description"].notna()].copy()
screening_source_jobs["description"] = screening_source_jobs["description"].astype(str).str.strip()
screening_source_jobs = screening_source_jobs[screening_source_jobs["description"] != ""].copy()
screening_source_jobs = screening_source_jobs.reset_index(drop=True)

if screening_source_jobs.empty:
    raise ValueError("The scraped jobs JSONL file contains no usable job descriptions for LLM screening.")

client = OpenAI()

screening_instructions = """
You classify whether a job posting is truly for an AI Engineering role.

AI Engineering definition:
- AI engineering means building applications on top of foundation models.
- Traditional ML engineering focuses on building, training, or tuning models; AI engineering primarily leverages existing models.
- An AI Engineer is not mainly a model researcher and does not primarily build models from scratch.
- AI engineering is closer to product and application engineering than to core ML research.

Decision rules:
- Set is_ai_engineering_role to true when the main responsibility is building product or application features on top of foundation models or LLMs.
- Set is_ai_engineering_role to false when the role is mainly traditional software engineering, data science, analytics, ML research, model training, classical ML engineering, MLOps or platform work, or something else where AI application work is not the core responsibility.
- If the posting is ambiguous or unclear, set is_ai_engineering_role to false.
""".strip()

screening_schema = {
    "type": "object",
    "properties": {
        "is_ai_engineering_role": {"type": "boolean"},
        "reason": {"type": "string"},
    },
    "required": ["is_ai_engineering_role", "reason"],
    "additionalProperties": False,
}

max_description_chars = 8000
screening_results = []

for i, (_, job) in enumerate(screening_source_jobs.iterrows(), start=1):
    title = "" if pd.isna(job.get("title")) else str(job.get("title")).strip()
    company = "" if pd.isna(job.get("company")) else str(job.get("company")).strip()
    job_url = "" if pd.isna(job.get("job_url")) else str(job.get("job_url")).strip()
    description = str(job.get("description", "")).strip()

    description_for_prompt = description[:max_description_chars]
    if len(description) > max_description_chars:
        description_for_prompt += "\n\n[Description truncated for screening.]"

    print(f"Screening job {i}/{len(screening_source_jobs)}: {title} @ {company}")

    response = client.responses.create(
        model="gpt-5.4-nano",
        instructions=screening_instructions,
        input=f"""Classify this job posting.\n\nTitle: {title}\nCompany: {company}\nURL: {job_url}\n\nDescription:\n{description_for_prompt}""",
        max_output_tokens=120,
        text={
            "format": {
                "type": "json_schema",
                "name": "ai_engineering_job_screening",
                "schema": screening_schema,
                "strict": True,
            },
            "verbosity": "low",
        },
    )

    result = json.loads(response.output_text)
    is_ai_engineering_role = bool(result.get("is_ai_engineering_role", False))
    reason = str(result.get("reason", "")).strip()

    screening_results.append(
        {
            "is_ai_engineering_role": is_ai_engineering_role,
            "llm_reason": reason,
        }
    )

screening_results_df = pd.DataFrame(screening_results)
screened_jobs = pd.concat([screening_source_jobs, screening_results_df], axis=1)
screened_jobs["is_ai_engineering_role"] = screened_jobs["is_ai_engineering_role"].fillna(False)
screened_jobs["llm_reason"] = screened_jobs["llm_reason"].fillna("")

ai_engineer_jobs = screened_jobs[screened_jobs["is_ai_engineering_role"]].copy()
non_ai_engineer_jobs = screened_jobs[~screened_jobs["is_ai_engineering_role"]].copy()
ai_engineer_jobs.to_json(llm_filtered_jsonl_path, orient='records', lines=True, force_ascii=False)

print(f"Saved LLM-filtered jobs to: {llm_filtered_jsonl_path.resolve()}")
print(f"Jobs screened by LLM: {len(screened_jobs)}")
print(f"Jobs classified as AI engineering roles: {len(ai_engineer_jobs)}")
print(f"Jobs classified as not AI engineering or unclear: {len(non_ai_engineer_jobs)}")

screening_results_to_show = screened_jobs[["title", "company", "is_ai_engineering_role", "llm_reason", "job_url"]].copy()
screening_results_to_show["job_url"] = screening_results_to_show["job_url"].apply(
    lambda url: f'<a href=\"{url}\" target=\"_blank\">{url}</a>' if pd.notna(url) and str(url).strip() else ''
)

display(HTML(screening_results_to_show.to_html(escape=False, index=False)))
